# PPI Style Playground

Use the controls below to iterate on presentation style. This focuses on **ribbon + surface + stick** combinations,
with adjustable colors, opacity, and thickness.


## 1) Install dependencies (run once if needed)


In [33]:
# Uncomment if needed:
# %pip install py3Dmol ipywidgets requests


## 2) Load a structure
Loads by PDB ID; falls back to a tiny demo if offline.


In [34]:
import requests
import py3Dmol
import ipywidgets as widgets
from IPython.display import display

DEMO_PDB = '''\
ATOM      1  N   ALA A   1      11.104  13.207  10.000  1.00 20.00           N
ATOM      2  CA  ALA A   1      12.560  13.407  10.000  1.00 20.00           C
ATOM      3  C   ALA A   1      13.000  14.800  10.500  1.00 20.00           C
ATOM      4  O   ALA A   1      12.400  15.800  10.300  1.00 20.00           O
ATOM      5  CB  ALA A   1      13.200  12.200  10.600  1.00 20.00           C
ATOM      6  N   VAL B   1      15.100  13.000  10.000  1.00 20.00           N
ATOM      7  CA  VAL B   1      16.500  13.200  10.000  1.00 20.00           C
ATOM      8  C   VAL B   1      16.900  14.600  10.500  1.00 20.00           C
ATOM      9  O   VAL B   1      16.200  15.600  10.400  1.00 20.00           O
ATOM     10  CB  VAL B   1      17.100  12.000  10.600  1.00 20.00           C
TER
END
'''

def load_pdb_from_id(pdb_id: str) -> str:
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return r.text

def get_structure(pdb_id: str = '4HHB') -> str:
    try:
        return load_pdb_from_id(pdb_id)
    except Exception:
        return DEMO_PDB

structure_text = get_structure('4HHB')


## 3) Interactive controls
Use the sliders/dropdowns to explore looks.


In [35]:
def render(
    style_mode='Ribbon',
    ribbon_color='spectrum',
    ribbon_hex='#d4af37',
    ribbon_thickness=0.5,
    ribbon_opacity=0.95,
    surface_opacity=0.35,
    surface_color='#ffffff',
    stick_radius=0.18,
    stick_color='element',
    bg_color='#0a0f14',
    add_sticks=False,
):
    view = py3Dmol.view(width=800, height=500)
    view.addModel(structure_text, 'pdb')
    view.setBackgroundColor(bg_color)

    color_value = ribbon_hex if ribbon_color == 'custom' else ribbon_color

    if style_mode in ['Ribbon', 'Ribbon + Surface']:
        view.setStyle({
            'cartoon': {
                'color': color_value,
                'thickness': float(ribbon_thickness),
                'opacity': float(ribbon_opacity)
            }
        })
    elif style_mode == 'Surface only':
        view.setStyle({'surface': {'opacity': float(surface_opacity), 'color': surface_color}})
    elif style_mode == 'Sticks only':
        view.setStyle({'stick': {'radius': float(stick_radius), 'color': stick_color}})

    if style_mode == 'Ribbon + Surface':
        view.addSurface(py3Dmol.VDW, {'opacity': float(surface_opacity), 'color': surface_color})

    if add_sticks and style_mode != 'Sticks only':
        view.addStyle({'stick': {'radius': float(stick_radius), 'color': stick_color}})

    view.zoomTo()
    return view.show()

controls = {
    'style_mode': widgets.Dropdown(
        options=['Ribbon', 'Ribbon + Surface', 'Surface only', 'Sticks only'],
        value='Ribbon',
        description='Mode'
    ),
    'ribbon_color': widgets.Dropdown(
        options=['spectrum', 'chain', 'element', 'white', 'black', 'custom'],
        value='spectrum',
        description='Ribbon color'
    ),
    'ribbon_hex': widgets.ColorPicker(
        value='#d4af37',
        description='Custom color'
    ),
    'ribbon_thickness': widgets.FloatSlider(
        value=0.5, min=0.2, max=1.2, step=0.05, description='Thickness'
    ),
    'ribbon_opacity': widgets.FloatSlider(
        value=0.95, min=0.2, max=1.0, step=0.05, description='Opacity'
    ),
    'surface_opacity': widgets.FloatSlider(
        value=0.35, min=0.05, max=1.0, step=0.05, description='Surface opacity'
    ),
    'surface_color': widgets.ColorPicker(
        value='#ffffff',
        description='Surface color'
    ),
    'stick_radius': widgets.FloatSlider(
        value=0.18, min=0.05, max=0.5, step=0.01, description='Stick radius'
    ),
    'stick_color': widgets.Dropdown(
        options=['element', 'white', 'black', 'spectrum', 'chain'],
        value='element',
        description='Stick color'
    ),
    'bg_color': widgets.ColorPicker(
        value='#0a0f14',
        description='Background'
    ),
    'add_sticks': widgets.Checkbox(
        value=False,
        description='Add sticks overlay'
    ),
}

ui = widgets.VBox([
    widgets.HBox([controls['style_mode'], controls['bg_color']]),
    widgets.HBox([controls['ribbon_color'], controls['ribbon_hex']]),
    widgets.HBox([controls['ribbon_thickness'], controls['ribbon_opacity']]),
    widgets.HBox([controls['surface_opacity'], controls['surface_color']]),
    widgets.HBox([controls['stick_radius'], controls['stick_color'], controls['add_sticks']]),
])

out = widgets.interactive_output(render, {k: v for k, v in controls.items()})
display(ui, out)


Output()